# Air quality in SNCF stations: data exploration (PySpark)

Hourly PM10 / PM2.5 readings from four SNCF station sensors in Île-de-France.
Source: https://ressources.data.sncf.com/explore/dataset/qualite-de-lair/information/

This notebook is a **Spark learning exercise**: every transformation and aggregation runs in Spark (DataFrame API / window functions).
The data is small enough for pandas, but the point is to write it the way it would be written for a much bigger dataset.
Pandas and matplotlib are used **only** to draw charts from tiny, already-aggregated results (`.toPandas()` on at most a few thousand rows), because Spark has no plotting.

Run it in the cluster's JupyterLab (`http://localhost:8888`, see `spark/README.md`) so it uses the standalone cluster. Copy it into `spark/apps/`, the kernel working directory.
Set `SPARK_MASTER=local[*]` to run without the cluster.

Sections: 1. Session and load · 2. Structure · 3. Column profile · 4. Data quality · 5. Patterns · 6. Summary and next steps.

## 1. Session and load

In [1]:
import os
import datetime as dt
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

MASTER = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
# Must be a path every executor can see: /opt/spark-data is mounted on master AND all workers.
DATA = os.environ.get("DATA_PATH", "/opt/spark-data/qualite-de-lair.csv")

spark = (SparkSession.builder
         .appName("air-quality-exploration")
         .master(MASTER)
         .config("spark.sql.session.timeZone", "UTC")      # all timestamps are UTC; convert explicitly when needed
         .config("spark.sql.shuffle.partitions", "8")      # default is 200: far too many for ~200k rows
         .config("spark.executor.memory", "1g")            # workers are capped at 2 GB
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/27 10:30:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/27 10:30:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.2.0 | master: spark://spark-master:7077 | UI: http://spark-master:4041


The file is `;`-delimited, has a header and starts with a UTF-8 BOM.
We give an **explicit schema** instead of `inferSchema`: it avoids an extra pass over the file, is stable if the data changes, and sidesteps the BOM in the first header name (columns are matched by position). Spark logs a harmless `CSV header does not conform to the schema` warning because we rename the French header columns on purpose.
The timestamp is read as a string and parsed with `try_to_timestamp`: Spark 4 runs in ANSI mode by default, so plain `to_timestamp` would *raise* on a malformed value instead of returning null.

In [ ]:
schema = StructType([
    StructField("code", StringType()),
    StructField("ts_raw", StringType()),
    StructField("PM10", DoubleType()),
    StructField("PM25", DoubleType()),
    StructField("station", StringType()),
    StructField("station_id", StringType()),
])

df = (spark.read.csv(DATA, sep=";", header=True, schema=schema)
      .withColumn("ts", F.try_to_timestamp("ts_raw", F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX")))
      .select("code", "station", "station_id", "ts", "PM10", "PM25", "ts_raw")
      .cache())                       # reused by almost every cell below
df.printSchema()
df.drop("ts_raw").show(5, truncate=False)

root
 |-- code: string (nullable = true)
 |-- station: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- PM10: double (nullable = true)
 |-- PM25: double (nullable = true)
 |-- ts_raw: string (nullable = true)



26/11/27 10:31:08 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/27 10:31:23 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/27 10:31:38 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


Spark transformations are lazy: nothing has run yet except schema resolution. `.cache()` is also lazy, so the first action below reads the CSV and fills the cache.

## 2. Structure

In [ ]:
n = df.count()
print(f"rows={n:,}  cols={len(df.columns) - 1}")
print("unparseable timestamps:", df.filter(F.col("ts").isNull()).count())
df.agg(F.min("ts").alias("first"), F.max("ts").alias("last")).show(truncate=False)
print("duplicate (station_id, ts):", df.groupBy("station_id", "ts").count().filter("count > 1").count())
print("timestamps not on the hour:", df.filter((F.minute("ts") != 0) | (F.second("ts") != 0)).count())
df.select("code", "station", "station_id").distinct().orderBy("code").show(truncate=False)

`code`, `station` and `station_id` form 4 distinct triples, so they are 1:1:1 and redundant with each other.
Spark gives no row-order guarantee, so unlike pandas there is nothing to check about the order of the file: any time-ordered logic must use an explicit `orderBy` / window.

In [ ]:
stations = (df.groupBy("code", "station")
              .agg(F.count("*").alias("rows"), F.min("ts").alias("first"), F.max("ts").alias("last"))
              # every expected hour must exist as a row; gaps are in the VALUES, not the timestamps
              .withColumn("expected_hours", ((F.unix_timestamp("last") - F.unix_timestamp("first")) / 3600 + 1).cast("long"))
              .withColumn("missing_rows", F.col("expected_hours") - F.col("rows"))
              .orderBy("code"))
stations.show(truncate=False)

Column roles: `ts` is the time key, `PM10` and `PM25` (µg/m³) are the metrics, the other three identify the station.
The station set changes over time (Neuilly from 2023, La Défense from 2025), so cross-station aggregates change composition.

## 3. Column profile

In [ ]:
pm = ["PM10", "PM25"]
df.select(*pm).summary("count", "mean", "stddev", "min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max").show()

In [ ]:
def col_stats(c):
    x = F.col(c)
    return [F.round(F.avg(x.isNull().cast("int")) * 100, 2).alias(f"{c}_null_%"),
            F.round(F.skewness(x), 2).alias(f"{c}_skew"),
            F.sum((x == 0).cast("int")).alias(f"{c}_zeros"),
            F.sum((x < 0).cast("int")).alias(f"{c}_negatives")]

df.agg(*col_stats("PM10"), *col_stats("PM25")).show(vertical=True)

In [ ]:
(df.groupBy("station")
   .agg(*[f(c).alias(f"{c}_{n}") for c in pm
          for n, f in [("mean", lambda c: F.round(F.mean(c), 1)),
                       ("median", lambda c: F.round(F.median(c), 1)),
                       ("p95", lambda c: F.round(F.percentile_approx(c, 0.95), 1)),
                       ("max", F.max)]])
   .orderBy("station")
   .show(truncate=False))

In [ ]:
df.agg(F.round(F.corr("PM10", "PM25"), 3).alias("corr_PM10_PM25")).show()
# PM25/PM10 ratio. try_divide returns null instead of raising on PM10 = 0 (ANSI mode).
(df.filter(F.col("PM10") > 0)
   .select(F.try_divide("PM25", "PM10").alias("PM25/PM10"))
   .summary("count", "mean", "25%", "50%", "75%")
   .show())

Histograms: Spark does the binning (`floor(x / width) * width` then `groupBy`), and only the ~80 bin counts per column reach pandas for drawing.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, c in zip(ax, pm):
    cap = df.approxQuantile(c, [0.995], 0.001)[0]
    width = cap / 80
    h = (df.filter(F.col(c).between(0, cap))
           .groupBy((F.floor(F.col(c) / width) * width).alias("bin")).count()
           .orderBy("bin").toPandas())
    a.bar(h["bin"], h["count"], width=width, align="edge")
    a.set(title=f"{c} (0 .. p99.5)", xlabel="µg/m³")
plt.tight_layout()

## 4. Data quality

### 4.1 Completeness: nulls come in long outages

In [ ]:
(df.groupBy("station")
   .agg(F.round(F.avg(F.col("PM10").isNull().cast("int")) * 100, 1).alias("PM10_null_%"),
        F.round(F.avg(F.col("PM25").isNull().cast("int")) * 100, 1).alias("PM25_null_%"))
   .unionByName(df.agg(F.lit("ALL").alias("station"),
                       F.round(F.avg(F.col("PM10").isNull().cast("int")) * 100, 1).alias("PM10_null_%"),
                       F.round(F.avg(F.col("PM25").isNull().cast("int")) * 100, 1).alias("PM25_null_%")))
   .show(truncate=False))

To measure how long the null stretches are we need **runs** of consecutive hours with the same state (the classic "gaps and islands" problem).
The pattern: order each station's rows with a window, flag the rows where the state differs from the previous row (`lag`), and take a running `sum` of those flags as a run id.
Windows are partitioned by `station`, so Spark processes one partition per station (4 here). That is fine for this exercise, but it is the part that would not scale with few, huge keys.

In [ ]:
def runs(sdf, state):
    # Runs of identical `state` per station. Returns: station, run_id, state, len, start, end.
    w = Window.partitionBy("station").orderBy("ts")
    x = sdf.select("station", "ts", state.alias("state"))
    x = x.withColumn("chg", F.coalesce((F.col("state") != F.lag("state").over(w)).cast("int"), F.lit(1)))
    x = x.withColumn("run_id", F.sum("chg").over(w))
    return (x.groupBy("station", "run_id", "state")
             .agg(F.count("*").alias("len"), F.min("ts").alias("start"), F.max("ts").alias("end")))

null_runs = runs(df, F.col("PM10").isNull()).filter("state").cache()
(null_runs.groupBy("station")
    .agg(F.count("*").alias("null_runs"),
         F.median("len").alias("median_len_h"),
         F.max("len").alias("max_len_h"),
         F.sum((F.col("len") >= 24).cast("int")).alias("runs_>=24h"),
         F.round(F.sum(F.when(F.col("len") >= 24, F.col("len")).otherwise(0)) / F.sum("len"), 2).alias("share_of_nulls_in_runs>=24h"))
    .orderBy("station")
    .show(truncate=False))

In [ ]:
station_names = [r["station"] for r in df.select("station").distinct().orderBy("station").collect()]

# Availability heatmap: % non-null PM10 per month and station (pivot done in Spark)
avail = (df.groupBy(F.date_format("ts", "yyyy-MM").alias("month"))
           .pivot("station", station_names)
           .agg(F.avg(F.col("PM10").isNotNull().cast("int")) * 100)
           .orderBy("month").toPandas().set_index("month"))

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(avail.values.astype(float), aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)
ax.set_xticks(range(avail.shape[1]), avail.columns, rotation=20, ha="right")
ax.set_yticks(range(0, len(avail), 6), avail.index[::6])
ax.grid(False); ax.set_title("PM10 availability (% of hours with a value) per month")
plt.colorbar(im, ax=ax, label="%"); plt.tight_layout()

In [ ]:
# PM10 null % per year and station
(df.groupBy(F.year("ts").alias("year"))
   .pivot("station", station_names)
   .agg(F.round(F.avg(F.col("PM10").isNull().cast("int")) * 100, 1))
   .orderBy("year").show(truncate=False))

In [ ]:
df.agg(F.sum((F.col("PM10").isNull() & F.col("PM25").isNull()).cast("int")).alias("both_null"),
       F.sum((F.col("PM10").isNull() & F.col("PM25").isNotNull()).cast("int")).alias("PM10_null_PM25_present"),
       F.sum((F.col("PM25").isNull() & F.col("PM10").isNotNull()).cast("int")).alias("PM25_null_PM10_present")).show()
(df.filter(F.col("PM10").isNull() & F.col("PM25").isNotNull())
   .groupBy("station").count().withColumnRenamed("count", "PM10_null_PM25_present").orderBy("station").show(truncate=False))

### 4.2 Validity: impossible and suspicious values

In [ ]:
x10, x25 = F.col("PM10"), F.col("PM25")
checks = {
    "PM10 < 0": x10 < 0,
    "PM25 < 0": x25 < 0,
    "PM25 > PM10 (impossible)": x25 > x10,
    "  of which PM10 <= 5": (x25 > x10) & (x10 <= 5),
    "PM10 == 5.0 (floor?)": x10 == 5.0,
    "PM25 == 5.0 (floor?)": x25 == 5.0,
    "PM10 > 1000": x10 > 1000,
    "PM10 > 2000": x10 > 2000,
    "PM25 > 500": x25 > 500,
}
df.agg(*[F.sum(cond.cast("int")).alias(name) for name, cond in checks.items()]).show(vertical=True, truncate=False)

In [ ]:
(df.groupBy("station")
   .agg(F.sum(((x10 < 0) | (x25 < 0)).cast("int")).alias("negatives"),
        F.sum((x10 == 5.0).cast("int")).alias("PM10==5.0"),
        F.sum((x25 == 5.0).cast("int")).alias("PM25==5.0"))
   .orderBy("station").show(truncate=False))
df.orderBy(F.desc("PM10")).select("station", "ts", "PM10", "PM25").show(10, truncate=False)

Stuck-sensor check: the same gaps-and-islands helper, with the PM10 *value* as the state (null rows dropped first).
A run of length > 1 means consecutive identical readings.

In [ ]:
same = runs(df.filter(F.col("PM10").isNotNull()), F.col("PM10")).cache()
(same.groupBy("station")
     .agg(F.sum(F.col("len") - 1).alias("repeated_values"), F.max("len").alias("longest_run"))
     .orderBy("station").show(truncate=False))
same.orderBy(F.desc("len")).select("station", F.col("state").alias("value"), "len", "start", "end").show(3, truncate=False)

### 4.3 Cleaning rules (proposed)
Applied below to a new DataFrame `clean`; thresholds are judgement calls and should be reviewed:
1. negative values → null
2. `PM25 > PM10` → both set to null (the pair is inconsistent)
3. `== 5.0` kept but flagged `at_floor` (likely a detection-limit substitute, not a real measurement)
4. values above a cap (PM10 > 1000 or PM25 > 500) → null

In [ ]:
neg = F.coalesce((x10 < 0) | (x25 < 0), F.lit(False))
inconsistent = F.coalesce(x25 > x10, F.lit(False))
huge = F.coalesce((x10 > 1000) | (x25 > 500), F.lit(False))
bad = neg | inconsistent | huge

clean = (df.withColumn("at_floor", F.coalesce((x10 == 5.0) | (x25 == 5.0), F.lit(False)))
           .withColumn("PM10", F.when(bad, F.lit(None).cast("double")).otherwise(x10))
           .withColumn("PM25", F.when(bad, F.lit(None).cast("double")).otherwise(x25))
           .drop("ts_raw").cache())

df.agg(F.sum(neg.cast("int")).alias("negatives"),
       F.sum(inconsistent.cast("int")).alias("PM25>PM10"),
       F.sum(huge.cast("int")).alias("above_cap"),
       F.sum(F.coalesce((x10 == 5.0) | (x25 == 5.0), F.lit(False)).cast("int")).alias("flagged_at_floor")).show()

def null_and_dist(sdf, label):
    return sdf.agg(F.lit(label).alias("version"),
                   F.round(F.avg(F.col("PM10").isNull().cast("int")) * 100, 2).alias("PM10_null_%"),
                   F.round(F.avg(F.col("PM25").isNull().cast("int")) * 100, 2).alias("PM25_null_%"),
                   F.round(F.mean("PM10"), 1).alias("PM10_mean"),
                   F.round(F.stddev("PM10"), 1).alias("PM10_std"),
                   F.max("PM10").alias("PM10_max"))
null_and_dist(df, "raw").unionByName(null_and_dist(clean, "clean")).show()

## 5. Patterns (cleaned data)
Timestamps are converted to Europe/Paris for the daily and weekly cycles (`from_utc_timestamp`; the session time zone stays UTC).
Medians are used because the distribution is heavily right-skewed. Each chart is one Spark aggregation (`groupBy` + `pivot`) whose result has a few dozen rows.

In [ ]:
clean = (clean.withColumn("local", F.from_utc_timestamp("ts", "Europe/Paris"))
              .withColumn("hour", F.hour("local"))
              .withColumn("dow", F.weekday("local"))          # 0 = Monday ... 6 = Sunday
              .withColumn("month", F.month("local"))
              .withColumn("year", F.year("local"))
              .cache())

def median_by(col):
    return (clean.groupBy(col).pivot("station", station_names)
                 .agg(F.median("PM10")).orderBy(col).toPandas().set_index(col))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, (c, title) in zip(ax, [("hour", "Median PM10 by hour (Paris time)"),
                              ("dow", "Median PM10 by weekday (0 = Mon)"),
                              ("month", "Median PM10 by month")]):
    median_by(c).plot(ax=a, title=title)
    a.set_ylabel("µg/m³"); a.legend(fontsize=7)
plt.tight_layout()

In [ ]:
(clean.withColumn("period", F.when(F.col("dow") >= 5, "weekend").otherwise("weekday"))
      .groupBy("station").pivot("period", ["weekday", "weekend"]).agg(F.round(F.median("PM10"), 1))
      .withColumn("weekend_vs_weekday_%", F.round((F.col("weekend") / F.col("weekday") - 1) * 100, 0))
      .orderBy("station").show(truncate=False))

In [ ]:
# Yearly medians; only trust years with decent availability (see section 4.1)
print("median PM10 per year:")
clean.groupBy("year").pivot("station", station_names).agg(F.round(F.median("PM10"), 0)).orderBy("year").show(truncate=False)
print("availability % for the same cells:")
(clean.groupBy("year").pivot("station", station_names)
      .agg(F.round(F.avg(F.col("PM10").isNotNull().cast("int")) * 100, 0)).orderBy("year").show(truncate=False))

Daily median with a 7-day rolling mean, computed in Spark with a **range window** over the day number (days with fewer than 12 valid hours are dropped first).

In [ ]:
epoch = F.lit(dt.date(1970, 1, 1))
daily = (clean.groupBy("station", F.to_date("local").alias("day"))
              .agg(F.median("PM10").alias("med"), F.count("PM10").alias("n"))
              .filter("n >= 12")
              .withColumn("d", F.datediff("day", epoch)))
w7 = Window.partitionBy("station").orderBy("d").rangeBetween(-6, 0)
timeline = (daily.withColumn("roll7", F.avg("med").over(w7))
                 .select("station", "day", "roll7").orderBy("station", "day").toPandas())

fig, ax = plt.subplots(figsize=(13, 4))
for st, g in timeline.groupby("station"):
    ax.plot(g["day"], g["roll7"], label=st, lw=1)
ax.set(title="Daily median PM10, 7-day rolling mean", ylabel="µg/m³"); ax.legend(); plt.tight_layout()

### 5.1 Hour × day heatmap (one station at a time)
`plot_heatmap(station_id)` takes a station id (`Id_Gare`) and draws **X = hour of day (Paris time), Y = calendar day, colour = PM10 level** (cleaned data) at exactly 1280×720 px. Grey cells are missing values, so sensor outages show up as grey bands.

It is split in two so the plot is only made when asked:
1. `heatmap_data(station_id, start, end)` is the **Spark** part. It filters the cleaned data to that one station, pivots the 24 hours into columns, and left-joins a calendar built with `sequence(first_day, last_day)` so days without any reading still get a (grey) row. Only that small matrix (≈ 3,000 rows × 24) leaves Spark.
2. `plot_heatmap(...)` calls it and draws with matplotlib. `start` / `end` (`"YYYY-MM-DD"`) zoom on a period, `save_as="file.png"` also writes the PNG.

The ids are in the table from section 2; below they are looked up by station code.

In [ ]:
W, H, DPI = 1280, 720, 100                                   # output size in pixels
get_ipython().run_line_magic("config", "InlineBackend.print_figure_kwargs = {'bbox_inches': None}")   # keep the exact size inline

station_ids = {r["code"]: r["station_id"] for r in df.select("code", "station_id").distinct().collect()}
station_names_by_id = {r["station_id"]: r["station"] for r in df.select("station_id", "station").distinct().collect()}
print(station_ids)

HOURS = list(range(24))
HOUR_COLS = [str(h) for h in HOURS]


def heatmap_data(station_id, start=None, end=None):
    # Spark: one row per calendar day, 24 columns (average PM10 per Paris hour). Missing hours/days are null.
    one = clean.filter(F.col("station_id") == station_id)
    day = F.to_date("local")
    first_day, last_day = one.agg(F.min(day), F.max(day)).first()
    first_day = max(first_day, dt.date.fromisoformat(start)) if start else first_day
    last_day = min(last_day, dt.date.fromisoformat(end)) if end else last_day
    calendar = spark.range(1).select(F.explode(F.sequence(F.lit(first_day), F.lit(last_day))).alias("day"))
    per_day = (one.groupBy(day.alias("day")).pivot("hour", HOURS).agg(F.avg("PM10")))
    return calendar.join(per_day, "day", "left").orderBy("day").toPandas()


def plot_heatmap(station_id, start=None, end=None, vmax=None, save_as=None):
    import numpy as np
    data = heatmap_data(station_id, start, end)
    if vmax is None:   # colour scale clipped at the station's own 99th percentile
        vmax = clean.filter(F.col("station_id") == station_id).approxQuantile("PM10", [0.99], 0.001)[0]
    z = np.ma.masked_invalid(data[HOUR_COLS].to_numpy(dtype=float))
    cmap = plt.get_cmap("magma_r").copy(); cmap.set_bad("#d9d9d9")

    fig, ax = plt.subplots(figsize=(W / DPI, H / DPI), dpi=DPI)
    im = ax.imshow(z, aspect="auto", cmap=cmap, vmin=0, vmax=vmax, interpolation="nearest")
    days = list(data["day"])
    if len(days) > 400:      # long period: one tick every Jan / Jul
        ticks = [i for i, d in enumerate(days) if d.day == 1 and d.month in (1, 7)]
        labels = [f"{days[i]:%Y-%m}" for i in ticks]
    else:                    # short period: one tick a week
        ticks = list(range(0, len(days), 7))
        labels = [str(days[i]) for i in ticks]
    ax.set_xticks(range(24), [str(h) for h in HOURS], fontsize=8)
    ax.set_yticks(ticks, labels, fontsize=8)
    ax.set(xlabel="hour of day (Paris time)", ylabel="day",
           title=f"PM10 by hour and day: {station_names_by_id.get(station_id, station_id)}  ({days[0]} to {days[-1]})")
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="PM10 µg/m³ (grey = no data)", pad=.015)
    fig.subplots_adjust(left=.09, right=.96, top=.93, bottom=.09)       # fixed margins: the canvas stays 1280x720
    if save_as:
        fig.savefig(save_as, dpi=DPI)
    return fig

In [ ]:
station_names_by_id

In [ ]:
plot_heatmap(station_ids["BDE"]);          # whole period, Avenue Foch

In [ ]:
plot_heatmap(station_ids["NPT"], start="2026-10-01");      # zoom: Neuilly, since 1 October 2026

Reading the full-period chart: the colour switches abruptly at ~07:00 Paris time (the night trough ends), and the long grey bands are the multi-month outages found in section 4.1.
In the zoomed one every row is a readable day: night trough, rush-hour bands and lighter weekend rows. A grey strip at the last row is the partial edge day (the data ends at 23:00 UTC, which is 00:00 Paris time of the next day).
Try the other stations with `plot_heatmap(station_ids["BDE"])` (Sevran) and `plot_heatmap(station_ids["DGA"])` (La Défense).

## 6. Summary and next steps

**Findings**
- 206,904 rows, 4 stations, hourly grain, no duplicate keys and no missing timestamps (Avenue Foch / Sevran 2018–, Neuilly 2023–, La Défense 2025–).
- ~15% of PM10 and ~16% of PM25 are null, mostly in outages of 24 h or more (up to ~157 days), and the null rate swings strongly by station and year.
- Values need cleaning: negatives, PM25 > PM10, a 5.0 floor value, extreme spikes, and a stuck sensor at Sevran (PM10 = 5.0 for 121 consecutive hourly readings, 2024-12-27 08:00 to 2025-01-01 08:00 UTC).
- The hour × day heatmap shows the same structure at a glance: a clean low-PM night trough that ends abruptly around 07:00 Paris time, and the outages as grey horizontal bands.
- Strong daily and weekly cycles: low at night, peaks at rush hour, weekends 17–46% lower depending on the station (Sevran least, Neuilly most). This looks like a rail-traffic signal, not outdoor air.

**Spark takeaways**
- Explicit schema + `try_*` functions: faster reads and no surprise failures under ANSI mode.
- `cache()` the DataFrame reused by many actions; check the *Storage* tab of the Application UI (`http://localhost:4040`).
- Window functions (`lag`, running `sum`, range windows) replace row-by-row pandas logic. Open the *SQL* and *Stages* tabs for the run-detection cells to see the shuffle and the single-partition-per-station effect. Past runs are in the History Server (`http://localhost:18080`).
- Only the final, tiny aggregates were converted with `.toPandas()`, and only to plot.

**Next steps**
1. Review and finalize the cleaning thresholds, then port them to a standalone Spark job in `src/pipe/` (submitted with `spark-submit`).
2. Write `clean` as partitioned Parquet (`.write.partitionBy("station", "year")`) and compare read times against the CSV.
3. Pick a valid analysis window per station from the availability heatmap.
4. Investigate spike dates (Sevran 2023-11-04, La Défense 2025-10-15) to separate real events from sensor faults.
5. Join an external station table if spatial analysis is needed (the data has no coordinates).